In [ ]:
# ── Imports and setup ──

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import umap
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
import os
warnings.filterwarnings('ignore')

torch.set_num_threads(torch.get_num_threads())
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
OUT = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT, exist_ok=True)

print(f"PyTorch version : {torch.__version__}")
print(f"Threads         : {torch.get_num_threads()}")
print(f"CUDA available  : {torch.cuda.is_available()}  (CPU-only run expected)")

In [ ]:
# ── Load and align CLR matrix with metadata ──

clr = pd.read_csv(os.path.join(OUT, "preprocessed_otu_clr.csv"), index_col=0)
meta = pd.read_csv(os.path.join(OUT, "merged_metadata.csv"))

meta = meta.set_index('sample_uid')
meta = meta.loc[clr.index]

print(f"CLR matrix          : {clr.shape}  (samples x OTUs)")
print(f"Metadata aligned    : {meta.shape}")
print(f"Studies             : {meta['study'].nunique()}")
print(f"Index match check   : {(clr.index == meta.index).all()}")
print(f"NaN in CLR          : {clr.isna().sum().sum()}")
print(f"\nTreatment counts:\n{meta['treatment'].value_counts()}")
print(f"\nTimepoint counts:\n{meta['timepoint'].value_counts()}")

In [ ]:
# ── Convert CLR matrix to tensors ──

X = torch.tensor(clr.values, dtype=torch.float32)
input_dim = X.shape[1]

dataset = TensorDataset(X)

n_val   = int(0.2 * len(dataset))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)

print(f"Input dim           : {input_dim}")
print(f"Train samples       : {n_train}  |  Val samples: {n_val}")
print(f"Train batches       : {len(train_loader)}")
print(f"Val batches         : {len(val_loader)}")
print(f"X dtype             : {X.dtype}  |  X shape: {tuple(X.shape)}")

In [ ]:
# ── Autoencoder architecture ──
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(),
            nn.Linear(512, 256),       nn.ReLU(),
            nn.Linear(256, 128),       nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256),        nn.ReLU(),
            nn.Linear(256, 512),        nn.ReLU(),
            nn.Linear(512, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

model     = Autoencoder(input_dim, latent_dim=32)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters    : {n_params:,}")
print(f"\nEncoder:")
for name, p in model.named_parameters():
    if 'encoder' in name and 'weight' in name:
        print(f"  {name:35s} {tuple(p.shape)}")
print(f"\nDecoder:")
for name, p in model.named_parameters():
    if 'decoder' in name and 'weight' in name:
        print(f"  {name:35s} {tuple(p.shape)}")

In [ ]:
# ── Training loop (early stopping, patience=10) ──

EPOCHS   = 100
PATIENCE = 10

best_val_loss  = float('inf')
patience_count = 0
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    t_loss = 0.0
    for (xb,) in train_loader:
        optimizer.zero_grad()
        recon, _ = model(xb)
        loss = criterion(recon, xb)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * len(xb)
    t_loss /= n_train

    # --- validate ---
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for (xb,) in val_loader:
            recon, _ = model(xb)
            v_loss += criterion(recon, xb).item() * len(xb)
    v_loss /= n_val

    train_losses.append(t_loss)
    val_losses.append(v_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3d}  train={t_loss:.6f}  val={v_loss:.6f}  patience={patience_count}")

    # early stopping
    if v_loss < best_val_loss:
        best_val_loss  = v_loss
        patience_count = 0
        torch.save(model.state_dict(), os.path.join(OUT, "autoencoder_model.pt"))
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}  (best val_loss={best_val_loss:.6f})")
            break

print(f"\nBest val loss       : {best_val_loss:.6f}")
print(f"Total epochs run    : {len(train_losses)}")
print(f"Model saved to      : {OUT}\\autoencoder_model.pt")

In [ ]:
# ── Reconstruction loss curve ──

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train MSE', linewidth=1.5)
ax.plot(val_losses,   label='Val MSE',   linewidth=1.5, linestyle='--')
ax.axvline(x=len(train_losses)-1, color='grey', linestyle=':', linewidth=1, label=f'Early stop (epoch {len(train_losses)})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder Reconstruction Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "reconstruction_loss_curve.png"), dpi=150)
plt.show()
print("Loss curve saved.")


In [ ]:
# ── Extract latent representations ──

model.load_state_dict(torch.load(os.path.join(OUT, "autoencoder_model.pt"), weights_only=True))
model.eval()

with torch.no_grad():
    _, Z = model(X)

Z_np = Z.numpy()

latent_df = pd.DataFrame(
    Z_np,
    index=clr.index,
    columns=[f"latent_{i}" for i in range(32)]
)
latent_df.to_csv(os.path.join(OUT, "latent_representations.csv"))

print(f"Latent matrix       : {latent_df.shape}")
print(f"Saved to            : {OUT}\\latent_representations.csv")
print(f"\nLatent value range  : {Z_np.min():.4f}  to  {Z_np.max():.4f}")
print(f"Mean per dim (first 5): {Z_np.mean(axis=0)[:5].round(4)}")
print(f"Std per dim (first 5) : {Z_np.std(axis=0)[:5].round(4)}")

In [ ]:
# ── UMAP projection ──

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=SEED,
    verbose=False
)

umap_coords = reducer.fit_transform(Z_np)

umap_df = pd.DataFrame(
    umap_coords,
    index=clr.index,
    columns=['UMAP1', 'UMAP2']
)

for col in ['study', 'fiber_type', 'timepoint', 'treatment']:
    umap_df[col] = meta[col].values

umap_df.to_csv(os.path.join(OUT, "umap_embeddings.csv"))

print(f"UMAP embeddings     : {umap_df.shape}")
print(f"UMAP1 range         : {umap_coords[:,0].min():.2f}  to  {umap_coords[:,0].max():.2f}")
print(f"UMAP2 range         : {umap_coords[:,1].min():.2f}  to  {umap_coords[:,1].max():.2f}")
print(f"Saved to            : {OUT}\\umap_embeddings.csv")

In [ ]:
# ── UMAP visualisation ──

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in zip(
    axes,
    ['study', 'fiber_type', 'timepoint'],
    ['Study (batch)', 'Fiber Type', 'Timepoint']
):
    categories = sorted(umap_df[col].astype(str).unique())
    palette    = cm.get_cmap('tab20', len(categories))
    cat_map    = {c: i for i, c in enumerate(categories)}
    colors     = [palette(cat_map[c]) for c in umap_df[col].astype(str)]

    ax.scatter(umap_df['UMAP1'], umap_df['UMAP2'],
               c=colors, s=6, alpha=0.6, linewidths=0)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('UMAP1')
    ax.set_ylabel('UMAP2')

    handles = [plt.Line2D([0],[0], marker='o', color='w',
                          markerfacecolor=palette(cat_map[c]),
                          markersize=6, label=c)
               for c in categories]
    ax.legend(handles=handles, fontsize=6, loc='best',
              framealpha=0.5, ncol=2)

plt.suptitle('Autoencoder Latent Space — UMAP Projections', fontsize=13)
plt.tight_layout()

savepath = os.path.join(OUT, "umap_plots.png")
plt.savefig(savepath, dpi=150)
plt.show()
print("UMAP plots saved to: " + savepath)

In [ ]:
# ── K-means clustering (k=3–6, silhouette selection) ──

sil_scores = {}
for k in [3, 4, 5, 6]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(Z_np)
    sil = silhouette_score(Z_np, labels, sample_size=1000, random_state=SEED)
    sil_scores[k] = (sil, labels, km)
    print(f"k={k}  silhouette={sil:.4f}")

best_k = max(sil_scores, key=lambda k: sil_scores[k][0])
best_labels = sil_scores[best_k][1]
print(f"\nBest k              : {best_k}  (silhouette={sil_scores[best_k][0]:.4f})")

cluster_df = umap_df.copy()
cluster_df['kmeans_cluster'] = best_labels
for k in [3, 4, 5, 6]:
    cluster_df[f'kmeans_k{k}'] = sil_scores[k][1]

cluster_df.to_csv(os.path.join(OUT, "cluster_assignments.csv"))
print(f"Saved to            : {OUT}\\cluster_assignments.csv")

In [ ]:
# ── Cluster composition (k=5) ──

# Override to k=5 per pre-specified analysis plan
best_k      = 5
best_labels = sil_scores[5][1]
cluster_df['kmeans_cluster'] = best_labels

print(f"=== Using k=5 (pre-specified to match fiber type groups) ===")
print(f"Silhouette k=5: {sil_scores[5][0]:.4f}  |  k=6: {sil_scores[6][0]:.4f}  (difference: {sil_scores[6][0]-sil_scores[5][0]:.4f})\n")

print("--- Cluster size ---")
print(cluster_df['kmeans_cluster'].value_counts().sort_index())

print("\n--- By study ---")
print(cluster_df.groupby(['kmeans_cluster','study']).size().unstack(fill_value=0).to_string())

print("\n--- By treatment ---")
print(cluster_df.groupby(['kmeans_cluster','treatment']).size().unstack(fill_value=0).to_string())

print("\n--- By timepoint ---")
print(cluster_df.groupby(['kmeans_cluster','timepoint']).size().unstack(fill_value=0).to_string())

print("\n--- Top 3 fiber types per cluster ---")
for c in sorted(cluster_df['kmeans_cluster'].unique()):
    top = (cluster_df[cluster_df['kmeans_cluster']==c]['fiber_type']
           .value_counts().head(3))
    print(f"  Cluster {c}: {dict(top)}")

In [ ]:
# ── Summary and save ──

cluster_df.to_csv(os.path.join(OUT, "cluster_assignments.csv"))

summary = pd.DataFrame({
    'n_samples'      : cluster_df['kmeans_cluster'].value_counts().sort_index(),
    'pct_fiber'      : cluster_df.groupby('kmeans_cluster').apply(
                           lambda x: (x['treatment']=='fiber').mean()*100).round(1),
    'pct_control'    : cluster_df.groupby('kmeans_cluster').apply(
                           lambda x: (x['treatment']=='control').mean()*100).round(1),
    'n_studies'      : cluster_df.groupby('kmeans_cluster')['study'].nunique(),
    'top_fiber_type' : cluster_df.groupby('kmeans_cluster')['fiber_type'].agg(
                           lambda x: x.value_counts().index[0])
})

print("=== Final cluster summary (k=5) ===\n")
print(summary.to_string())
print(f"\nCluster assignments saved : {OUT}\\cluster_assignments.csv")
print("\nStage 4 complete. Outputs:")
print(f"  autoencoder_model.pt")
print(f"  latent_representations.csv   ({latent_df.shape[0]} samples x 32 dims)")
print(f"  umap_embeddings.csv          ({umap_df.shape[0]} samples x 2 UMAP dims + metadata)")
print(f"  reconstruction_loss_curve.png")
print(f"  umap_plots.png")
print(f"  cluster_assignments.csv      (k=3,4,5,6 all saved; k=5 is primary)")